In [1]:
import pandas as pd
from pathlib import Path

In [2]:
# 파일명과 해당 연도 정리
files = [
    ("../data/raw(원본)/2020_2021.csv", "2020", "2021"),
    ("../data/raw(원본)/2022_2023.csv", "2022", "2023"),
    ("../data/raw(원본)/2024_2025.csv", "2024", "2025")
]

# 정제된 데이터를 담을 빈 리스트
data_list = []

for file, year1, year2 in files:

    # CSV 불러오기
    temp = pd.read_csv(
        file,
        encoding="utf-8-sig",
        header=None
    )

    # 입출항구분 / 국가명 / 두 연도의 누계 '계'만 선택
    temp = temp.iloc[:, [0, 1, 5, 11]].copy()

    # 컬럼명 변경
    temp.columns = [
        "입출항구분",
        "국가명",
        year1,
        year2
    ]

    # 위쪽 헤더 3행 제거
    temp = temp.iloc[3:].copy()

    # 입출항구분 빈칸 채우기
    temp["입출항구분"] = temp["입출항구분"].ffill()

    # 국가명이 없는 합계 / 소계 제거
    temp = temp.dropna(subset=["국가명"])

    # 필요한 4개 구분만 남기기
    temp = temp[
        temp["입출항구분"].isin([
            "입항",
            "출항",
            "입항환적",
            "출항환적"
        ])
    ].copy()

    # 물동량 숫자형 변환
    temp[year1] = pd.to_numeric(temp[year1], errors="coerce")
    temp[year2] = pd.to_numeric(temp[year2], errors="coerce")

    # 물동량이 없는 경우 0으로 처리
    temp[[year1, year2]] = temp[[year1, year2]].fillna(0)

    # 연도를 세로 형태로 변환
    temp_long = temp.melt(
        id_vars=["입출항구분", "국가명"],
        value_vars=[year1, year2],
        var_name="연도",
        value_name="물동량"
    )

    # 연도를 숫자로 변환
    temp_long["연도"] = temp_long["연도"].astype(int)

    # 리스트에 저장
    data_list.append(temp_long)

In [3]:
# 2020~2025 데이터 합치기
df_all = pd.concat(
    data_list,
    ignore_index=True
)

df_all.head()

,입출항구분,국가명,연도,물동량
0,입항,가나,2020,774.0
1,입항,가봉,2020,9.0
2,입항,가이아나,2020,79.0
3,입항,감비아,2020,98.0
4,입항,과들루프,2020,0.0


In [4]:
# 연도별 데이터 개수 확인
df_all["연도"].value_counts().sort_index()

연도
2020    663
2021    663
2022    674
2023    674
2024    668
2025    668
Name: count, dtype: int64

In [5]:
# 결측치 확인
df_all.isna().sum()

입출항구분    0
국가명      0
연도       0
물동량      0
dtype: int64

In [6]:
# 입출항구분 확인
df_all["입출항구분"].value_counts()

입출항구분
출항환적    1048
출항       996
입항환적     984
입항       982
Name: count, dtype: int64

In [7]:
# 연도별 부산항 전체 물동량 확인
df_all.groupby("연도")["물동량"].sum()

연도
2020    21823961.00
2021    22706123.75
2022    22076784.50
2023    23152998.00
2024    24402020.00
2025    24882345.00
Name: 물동량, dtype: float64

In [8]:
# 수입
import_df = df_all[
    df_all['입출항구분'] == '입항'
].groupby(
    ['연도', '국가명']
)['물동량'].sum().reset_index()

# 수출
export_df = df_all[
    df_all['입출항구분'] == '출항'
].groupby(
    ['연도', '국가명']
)['물동량'].sum().reset_index()

# 환적
trans_df = df_all[
    df_all['입출항구분'].isin(['입항환적', '출항환적'])
].groupby(
    ['연도', '국가명']
)['물동량'].sum().reset_index()

# 전체 물동량
total_df = df_all.groupby(
    ['연도', '국가명']
)['물동량'].sum().reset_index()

In [9]:
import_df.head()

,연도,국가명,물동량
0,2020,가나,774.0
1,2020,가봉,9.0
2,2020,가이아나,79.0
3,2020,감비아,98.0
4,2020,과들루프,0.0


In [10]:
export_df.head()

,연도,국가명,물동량
0,2020,가나,3203.0
1,2020,가봉,16.0
2,2020,가이아나,73.0
3,2020,감비아,6.0
4,2020,과들루프,21.0


In [11]:
trans_df.head()

,연도,국가명,물동량
0,2020,가나,1798.5
1,2020,가봉,984.0
2,2020,가이아나,4860.0
3,2020,감비아,134.0
4,2020,과들루프,182.0


In [12]:
total_df.head()

,연도,국가명,물동량
0,2020,가나,5775.5
1,2020,가봉,1009.0
2,2020,가이아나,5012.0
3,2020,감비아,238.0
4,2020,과들루프,203.0


In [13]:
print('수입')
print(import_df.groupby('연도')['물동량'].sum())

print('\n수출')
print(export_df.groupby('연도')['물동량'].sum())

print('\n환적')
print(trans_df.groupby('연도')['물동량'].sum())

print('\n전체')
print(total_df.groupby('연도')['물동량'].sum())

수입
연도
2020    4852616.75
2021    5207989.50
2022    5133224.00
2023    5330591.50
2024    5409691.25
2025    5362957.75
Name: 물동량, dtype: float64

수출
연도
2020    4951218.75
2021    5225461.75
2022    5177749.75
2023    5413643.75
2024    5495144.75
2025    5422185.75
Name: 물동량, dtype: float64

환적
연도
2020    12020125.50
2021    12272672.50
2022    11765810.75
2023    12408762.75
2024    13497184.00
2025    14097201.50
Name: 물동량, dtype: float64

전체
연도
2020    21823961.00
2021    22706123.75
2022    22076784.50
2023    23152998.00
2024    24402020.00
2025    24882345.00
Name: 물동량, dtype: float64


In [14]:
# 1. 전체 정제 데이터
df_all.to_csv(
    "../data/processed(정제)/01_all_2020_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

In [15]:
# 2. 국가별 전체 물동량
total_df.to_csv(
    "../data/processed(정제)/02_total_2020_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

In [16]:
# 3. 국가별 수입
import_df.to_csv(
    "../data/processed(정제)/03_import_2020_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

In [17]:
# 4. 국가별 수출
export_df.to_csv(
    "../data/processed(정제)/04_export_2020_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

In [18]:
# 5. 국가별 환적
trans_df.to_csv(
    "../data/processed(정제)/05_transshipment_2020_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

In [19]:
# 각 데이터의 물동량 컬럼 이름 변경
import_master = import_df.rename(columns={'물동량': '수입'})
export_master = export_df.rename(columns={'물동량': '수출'})
trans_master = trans_df.rename(columns={'물동량': '환적'})

In [20]:
# 수입 + 수출
master_df = pd.merge(
    import_master,
    export_master,
    on=['연도', '국가명'],
    how='outer'
)

# 환적 추가
master_df = pd.merge(
    master_df,
    trans_master,
    on=['연도', '국가명'],
    how='outer'
)

In [21]:
# 없는 물동량은 0으로 처리
master_df[['수입', '수출', '환적']] = (
    master_df[['수입', '수출', '환적']]
    .fillna(0)
)

# 전체 물동량 계산
master_df['전체물동량'] = (
    master_df['수입']
    + master_df['수출']
    + master_df['환적']
)

In [22]:
# 연도 → 국가명 순으로 정렬
master_df = master_df.sort_values(
    ['연도', '국가명']
).reset_index(drop=True)

master_df.head(10)

,연도,국가명,수입,수출,환적,전체물동량
0,2020,가나,774.0,3203.0,1798.50,5775.50
1,2020,가봉,9.0,16.0,984.00,1009.00
2,2020,가이아나,79.0,73.0,4860.00,5012.00
3,2020,감비아,98.0,6.0,134.00,238.00
4,2020,과들루프,0.0,21.0,182.00,203.00
5,2020,과테말라,6007.0,2702.0,21203.75,29912.75
6,2020,괌,10166.0,2785.0,7258.25,20209.25
7,2020,그레나다,0.0,32.0,474.00,506.00
8,2020,그리스,13427.0,16723.0,16204.25,46354.25
9,2020,기니,886.0,170.0,227.00,1283.00


In [23]:
# 마스터 데이터의 연도별 전체 물동량 확인
master_df.groupby('연도')['전체물동량'].sum()

연도
2020    21823961.00
2021    22706123.75
2022    22076784.50
2023    23152998.00
2024    24402020.00
2025    24882345.00
Name: 전체물동량, dtype: float64

In [24]:
master_df.to_csv(
    "../data/processed(정제)/master_2020_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

In [25]:
# 2. 국가별 전체 물동량
total_df.to_csv(
    "../data/processed(정제)/02_total_2020_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

total_df.head(10)

,연도,국가명,물동량
0,2020,가나,5775.50
1,2020,가봉,1009.00
2,2020,가이아나,5012.00
3,2020,감비아,238.00
4,2020,과들루프,203.00
5,2020,과테말라,29912.75
6,2020,괌,20209.25
7,2020,그레나다,506.00
8,2020,그리스,46354.25
9,2020,기니,1283.00


NameError: name 'monthly_all' is not defined